# Before the Breach — Source-specific Bayesian Exploitation Model

**Experiment:** replace IBM's *any exploit code → EPSS* heuristic with a learned, source-specific evidence score. Four signals (`exploitdb`, `metasploit`, `nuclei`, `poc_github`) are *not* assumed to be equally predictive. A regularized Naive Bayes evidence model estimates each signal's likelihood ratio, then adjusts the snapshot EPSS log-odds.

This is **not a demonstrated win**: the notebook benchmarks against IBM's published exploit-first baseline on historical held-out quarters. Because 2025 Q1–Q3 have been examined in the project already, this is an exploratory robustness check, not an untouched test. 2026 KEV outcomes are not supplied; do not claim an official 2026 improvement.

**Point-in-time guard:** Uses only `scoreboard_2026.csv`-compatible columns. No current NIST CVSS/status, product/references metadata, post-attack descriptions, or current KEV flags as input features. KEV additions through 2025-12-31 are used **only as historical targets**. February 2025 is excluded because of significant source missingness. The 2025 EPSS model switch means this is still a domain-shift challenge.

Keep this notebook next to `data/` in your repo. Requires pandas, numpy, scipy, scikit-learn, matplotlib.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.special import logit, expit

DATA = Path('data') if Path('data').exists() else Path('../data')
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
FLAGS = ['exploitdb', 'metasploit', 'nuclei', 'poc_github']
HORIZON_DAYS = 90
LABEL_END = pd.Timestamp('2025-12-31')
K_FRAC = 500 / 72670

v = pd.read_csv(DATA/'vulnerabilities.csv', usecols=['cve_id','published'])
k = pd.read_csv(DATA/'known_exploited.csv', usecols=['cve_id','date_added']).drop_duplicates('cve_id')
s = pd.concat([pd.read_csv(DATA/'exploit_signals_2024.csv'),pd.read_csv(DATA/'exploit_signals_2025.csv')],ignore_index=True)
v['published'] = pd.to_datetime(v['published'], errors='coerce')
k['date_added'] = pd.to_datetime(k['date_added'], errors='coerce')

# Same population definition as official scoreboard: one observation at publication month end
cohort = v[v['published'].between('2024-01-01','2025-09-30')].copy()
cohort['snapshot_month'] = cohort['published'].dt.strftime('%Y-%m')
cohort = cohort[cohort['snapshot_month']!='2025-02'].copy()
s = s.rename(columns={'month':'snapshot_month'})
cohort = cohort.merge(s[['cve_id','snapshot_month','epss']+FLAGS], on=['cve_id','snapshot_month'], how='left', validate='one_to_one')
cohort = cohort.merge(k, on='cve_id', how='left', validate='one_to_one')
cohort['snapshot_date'] = pd.to_datetime(cohort['snapshot_month']+'-01') + pd.offsets.MonthEnd(0)
cohort = cohort[(cohort.date_added.isna())|(cohort.date_added>cohort.snapshot_date)].copy()
cohort = cohort[cohort.snapshot_date + pd.Timedelta(days=HORIZON_DAYS) <= LABEL_END].copy()
cohort['label'] = (cohort.date_added.notna() & (cohort.date_added > cohort.snapshot_date) & (cohort.date_added <= cohort.snapshot_date+pd.Timedelta(days=HORIZON_DAYS))).astype('int8')
cohort['epss_missing'] = cohort.epss.isna().astype('int8')
cohort['epss'] = cohort.epss.fillna(0.0).clip(0,1)
cohort[FLAGS] = cohort[FLAGS].fillna(0).astype('int8')
cohort['any_exploit'] = cohort[FLAGS].max(axis=1)
print('Cohort:',cohort.shape,'positives:',int(cohort.label.sum()))
display(cohort.groupby(cohort.snapshot_month.str[:4]).label.agg(['count','sum']))

Cohort: (71485, 13) positives: 70


,count,sum
snapshot_month,,
2024,39868,32
2025,31617,38


## Evidence model

For each exploit flag, we estimate a likelihood ratio:

\[ LR_i=\frac{P(\text{signal}_i\mid \text{future KEV})}{P(\text{signal}_i\mid \text{not added to KEV})} \]

Using *Beta-prior shrinkage* avoids gigantic weights from extremely rare signals and just ~tens of positive examples. We handle both flag present and absent. The total learned evidence adjusts historical EPSS log-odds, with a **predeclared** evidence strength of 1.0. This is a ranking score, **not a calibrated probability**.

The learner is different from *exploit available = rank first*, because it learns source-specific signal strength and allows sufficiently high EPSS to compete with weak exploit flags. We do not use the same KEV-positive row as both a feature and a target.

In [2]:
class SourceEvidenceRanker:
    def __init__(self, positive_prior=20., negative_prior=200., evidence_weight=1.0):
        self.positive_prior = positive_prior
        self.negative_prior = negative_prior
        self.evidence_weight = evidence_weight

    def fit(self, X, y):
        y = np.asarray(y).astype(bool)
        f = X[FLAGS].astype(int).to_numpy()
        assert y.sum() >= 1 and (~y).sum() >= 1
        base = f.mean(axis=0)
        p1 = (f[y].sum(axis=0) + self.positive_prior*base)/(y.sum()+self.positive_prior)
        p0 = (f[~y].sum(axis=0)+self.negative_prior*base)/((~y).sum()+self.negative_prior)
        self.p1_ = np.clip(p1, 1e-6, 1-1e-6)
        self.p0_ = np.clip(p0, 1e-6, 1-1e-6)
        self.evidence_table_ = pd.DataFrame({
            'source':FLAGS, 'P(signal|future_KEV)':self.p1_, 'P(signal|not_KEV)':self.p0_,
            'LR_when_present':self.p1_/self.p0_,
            'LR_when_absent':(1-self.p1_)/(1-self.p0_)
        })
        return self

    def score(self, X):
        f = X[FLAGS].astype(int).to_numpy()
        evidence = (f*np.log(self.p1_/self.p0_) + (1-f)*np.log((1-self.p1_)/(1-self.p0_))).sum(axis=1)
        # The public EPSS model is already ML and uses exploit-related info;
        # this addition can double-count signals, so validation is essential.
        epss = np.clip(X['epss'].to_numpy(dtype=float), 1e-5, 1-1e-5)
        return logit(epss) + self.evidence_weight*evidence

def rank_score_baseline(frame):
    # Lexicographic priority: any observed exploit source, then EPSS
    f = frame.any_exploit.to_numpy()
    e = frame.epss.to_numpy()
    order = np.lexsort((-e,-f))
    out = np.empty(len(frame),dtype=float)
    out[order] = np.arange(len(frame),0,-1)
    return out

def measure(frame, score, label, fold):
    y=frame.label.to_numpy()
    kk=max(1,round(len(frame)*K_FRAC))
    inds=np.argsort(-score,kind='stable')[:kk]
    return {'fold':fold,'model':label,'candidates':len(frame),'positives':int(y.sum()),
            'k':kk,'hits_at_k':int(y[inds].sum()), 'average_precision':average_precision_score(y,score),
            'roc_auc':roc_auc_score(y,score) if y.sum()>0 and y.sum()<len(y) else np.nan}

FOLDS=[('2025-Q1','2025-01-01','2025-04-01'),('2025-Q2','2025-04-01','2025-07-01'),('2025-Q3','2025-07-01','2025-10-01')]
results=[]
source_tables=[]
for name, start, end in FOLDS:
    st=pd.Timestamp(start)
    tr=cohort[cohort.snapshot_date + pd.Timedelta(days=HORIZON_DAYS) < st].copy()
    va=cohort[(cohort.snapshot_date >= st)&(cohort.snapshot_date < pd.Timestamp(end))].copy().reset_index(drop=True)
    if not len(va) or tr.label.sum()==0: raise ValueError('Empty fold or zero positives: '+name)
    learned=SourceEvidenceRanker().fit(tr,tr.label)
    source_tables.append((name,learned.evidence_table_))
    results.extend([measure(va,rank_score_baseline(va),'IBM-style exploit first -> EPSS',name),
                    measure(va,learned.score(va),'Bayesian source evidence + EPSS',name)])
    print(f'{name}: train {len(tr):,}, train events {tr.label.sum()}, validate {len(va):,}, validate events {va.label.sum()}')
folds_df=pd.DataFrame(results)
display(folds_df)
display(folds_df.groupby('model')[['hits_at_k','k']].sum().assign(precision=lambda d:d.hits_at_k/d.k))
print('Source likelihood ratios in last fold:')
display(source_tables[-1][1])

2025-Q1: train 28,839, train events 20, validate 8,249, validate events 11
2025-Q2: train 39,868, train events 32, validate 11,666, validate events 14
2025-Q3: train 48,117, train events 43, validate 11,702, validate events 13


,fold,model,candidates,positives,k,hits_at_k,average_precision,roc_auc
0,2025-Q1,IBM-style exploit first -> EPSS,8249,11,57,3,0.111329,0.719206
1,2025-Q1,Bayesian source evidence + EPSS,8249,11,57,3,0.114294,0.702487
2,2025-Q2,IBM-style exploit first -> EPSS,11666,14,80,6,0.135093,0.785420
3,2025-Q2,Bayesian source evidence + EPSS,11666,14,80,6,0.129652,0.784461
4,2025-Q3,IBM-style exploit first -> EPSS,11702,13,81,2,0.011307,0.833703
5,2025-Q3,Bayesian source evidence + EPSS,11702,13,81,0,0.007280,0.823996


,hits_at_k,k,precision
model,,,
Bayesian source evidence + EPSS,9,218,0.041284
IBM-style exploit first -> EPSS,11,218,0.050459


Source likelihood ratios in last fold:


,source,P(signal|future_KEV),P(signal|not_KEV),LR_when_present,LR_when_absent
0,exploitdb,0.016025,0.000458,35.010562,0.984426
1,metasploit,0.031918,0.000499,63.911746,0.968566
2,nuclei,0.112140,0.003100,36.174543,0.890621
3,poc_github,0.180640,0.018805,9.605846,0.835064


## Selection, 2026 ranking, and evidence explanations

Model selection intentionally favors the baseline: adopt the Bayesian challenger only if **aggregate hits increase and no quarter loses hits**. We do not claim an unbiased result since these historical folds were previously inspected. The generated files are **candidates** pending 2026 judge evaluation. The dataset lacks verified 2026 outcome labels.

`source_likelihood_ratios.csv` provides interpretable source-specific evidence for the dashboard. These are associations, not causal claims.

In [3]:
p=folds_df.pivot(index='fold',columns='model',values='hits_at_k')
base_name='IBM-style exploit first -> EPSS'
new_name='Bayesian source evidence + EPSS'
prefer_new=bool((p[new_name].sum() > p[base_name].sum()) and (p[new_name]>=p[base_name]).all())
print('Selected:',new_name if prefer_new else base_name)

final_model=SourceEvidenceRanker().fit(cohort,cohort.label)
sc=pd.read_csv(DATA/'scoreboard_2026.csv')
assert sc.cve_id.is_unique and len(sc)==72670
sc['epss']=sc.epss.fillna(0).clip(0,1)
sc[FLAGS]=sc[FLAGS].fillna(0).astype('int8')
sc['any_exploit']=sc[FLAGS].max(axis=1)
sc['baseline_score']=rank_score_baseline(sc)
sc['bayes_score']=final_model.score(sc)
sc['selected_score']=sc['bayes_score'] if prefer_new else sc['baseline_score']
for stem,column in [('baseline','baseline_score'),('bayesian','bayes_score'),('selected','selected_score')]:
    ids=sc.sort_values(column,ascending=False,kind='stable').head(500)[['cve_id']]
    assert len(ids)==500 and ids.cve_id.is_unique
    ids.to_csv(OUT/f'top500_{stem}.csv',index=False)
sc.sort_values('selected_score',ascending=False).to_csv(OUT/'ranked_cves_with_evidence.csv',index=False)
final_model.evidence_table_.to_csv(OUT/'source_likelihood_ratios.csv',index=False)
print('Wrote:',*[x.name for x in sorted(OUT.glob('*.csv'))],sep='\n  ')
display(sc.sort_values('selected_score',ascending=False).head(15)[['cve_id','epss']+FLAGS+['baseline_score','bayes_score']])

Selected: IBM-style exploit first -> EPSS
Wrote:
  granite_rerank_folds.csv
  granite_rerank_summary.csv
  granite_vector_ids.csv
  ranked_2026.csv
  ranked_cves_with_evidence.csv
  semantic_metadata.csv
  source_likelihood_ratios.csv
  top500_baseline.csv
  top500_bayesian.csv
  top500_granite_candidate.csv
  top500_selected.csv


,cve_id,epss,exploitdb,metasploit,nuclei,poc_github,baseline_score,bayes_score
62040,CVE-2026-8037,0.84793,0,0,1,1,72670.0,7.996093
1596,CVE-2022-25369,0.63500,0,0,1,0,72669.0,3.986857
2072,CVE-2024-45257,0.59548,0,1,0,0,72668.0,4.152406
40258,CVE-2026-49160,0.53829,0,0,0,1,72667.0,2.535236
2907,CVE-2025-14558,0.53599,1,1,0,1,72666.0,9.669525
9550,CVE-2026-10523,0.51870,0,0,0,1,72665.0,2.456611
2104,CVE-2024-51092,0.44112,0,1,0,0,72664.0,3.529121
24158,CVE-2026-27771,0.43068,0,0,1,1,72663.0,5.998559
25345,CVE-2026-29058,0.42990,0,1,0,0,72662.0,3.483480
20349,CVE-2026-2329,0.41137,0,1,0,0,72661.0,3.407434


## Interpretation and follow-up

- If the Bayesian challenger loses, do not put it in the submitted ranking just because it is novel. Keep its source-strength explanation as an analysis result.
- If it gains one hit in previously inspected folds, confirm on a genuinely new temporal validation cohort before claiming an improvement.
- A more ambitious follow-up is to obtain **historical (as-of-snapshot) software exposure / vendor and product context**, or a historical CVE-description feed. Current NIST fields in your bundled dataset are **not safe substitutes**.
- EPSS is a pre-existing model and probably already incorporates exploit-source evidence; multiplying evidence can double-count correlations. Compare only on future historical folds.
- The official IBM benchmark's top-500 counts cannot be inferred from historical fold performance.

### Relevant open source

- [joemolto/cve-exploit-prediction](https://github.com/joemolto/cve-exploit-prediction) — point-in-time evaluation, lexical TF-IDF sometimes outperforming embeddings.
- [ihassanrayan-max/patchpilot](https://github.com/ihassanrayan-max/patchpilot) — EPSS-anchored residual score with time-aligned evaluation.
- [scadastrangelove/kev_vs_epss](https://github.com/scadastrangelove/kev_vs_epss) — EPSS history, lead time, evidence timelines.
- [CERTCC/SSVC](https://github.com/CERTCC/SSVC) — practical context-aware remediation decisions (not a substitute for competition top-500 prediction).
